# Module 5: Integrated Executive Dashboard
## Aurora Finance | ML-Driven Decision Support — All Modules

**Purpose:** A single unified dashboard giving Aurora Finance's executive team
a consolidated view of all four ML modules:

| Panel | Source | Key Insight |
|---|---|---|
| **Corporate Finance** | Module 1 | Top projects by Risk-Adjusted Expected Value |
| **Banking** | Module 2 | Credit risk tier breakdown + Fraud alerts |
| **Financial Markets** | Module 3 | Portfolio allocation + Backtest equity curve |
| **Derivatives** | Module 4 | Option pricing comparison + VaR + Delta hedge |

**Output:** Interactive HTML dashboard + Excel summary report

In [1]:
import subprocess
pkgs = ["plotly","pandas","numpy","openpyxl","kaleido"]
subprocess.run(["pip","install"] + pkgs + ["-q"], check=True)
print("Dependencies ready.")

Dependencies ready.


In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os, warnings
warnings.filterwarnings("ignore")

_nb_dir  = os.path.dirname(os.path.abspath("module5_integrated_dashboard.ipynb"))
ROOT_DIR = os.path.dirname(_nb_dir)

# Module output paths
M1_DIR = os.path.join(ROOT_DIR, "Corporate Finance Module")
M2_DIR = os.path.join(ROOT_DIR, "Banking Module")
M3_DIR = os.path.join(ROOT_DIR, "Finance Markets Module")
M4_DIR = os.path.join(ROOT_DIR, "Derivatives Module")

print("Root:", ROOT_DIR)
for d in [M1_DIR, M2_DIR, M3_DIR, M4_DIR]:
    exists = "OK" if os.path.isdir(d) else "MISSING"
    print(f"  {os.path.basename(d)}: {exists}")

Root: D:\IIT Course\Sem3\iitb-capstone-project
  Corporate Finance Module: OK
  Banking Module: OK
  Finance Markets Module: OK
  Derivatives Module: OK


---
## 1. Load Module Outputs

In [3]:
# Module 1 — Corporate Finance
m1_ranked    = pd.read_csv(os.path.join(M1_DIR, "ranked_projects.csv"))

# Module 2 — Banking
m2_loans     = pd.read_csv(os.path.join(M2_DIR, "risk_scored_loans.csv"))
m2_fraud     = pd.read_csv(os.path.join(M2_DIR, "fraud_alerts.csv"))

# Module 3 — Financial Markets
m3_stocks    = pd.read_csv(os.path.join(M3_DIR, "stock_ranking.csv"))
m3_weights   = pd.read_csv(os.path.join(M3_DIR, "portfolio_weights.csv"))
m3_backtest  = pd.read_csv(os.path.join(M3_DIR, "backtest_returns.csv"), parse_dates=["date"])

# Module 4 — Derivatives
m4_pricing   = pd.read_csv(os.path.join(M4_DIR, "pricing_comparison.csv"))
m4_var       = pd.read_csv(os.path.join(M4_DIR, "var_results.csv"))
m4_hedge     = pd.read_csv(os.path.join(M4_DIR, "delta_hedge_table.csv"))

print("All data loaded successfully.")
print(f"  M1 projects  : {len(m1_ranked)} rows")
print(f"  M2 loans     : {len(m2_loans)} rows | fraud alerts: {len(m2_fraud)} rows")
print(f"  M3 stocks    : {len(m3_stocks)} rows | backtest: {len(m3_backtest)} rows")
print(f"  M4 pricing   : {len(m4_pricing)} rows | VaR: {len(m4_var)} rows")

All data loaded successfully.
  M1 projects  : 50 rows
  M2 loans     : 100 rows | fraud alerts: 200 rows
  M3 stocks    : 6 rows | backtest: 525 rows
  M4 pricing   : 3 rows | VaR: 4 rows


---
## 2. Panel 1 — Corporate Finance: Project Funding Decisions

In [4]:
# Top 10 projects by Risk-Adjusted EV
top10 = m1_ranked.sort_values("Risk_Adj_EV", ascending=False).head(10).copy()
top10["Label"] = top10["Project_ID"].astype(str).apply(lambda x: f"P{x}")
top10["EV_M"]  = top10["Risk_Adj_EV"] / 1e6   # in millions
top10["NPV_M"] = top10["NPV"] / 1e6

color_map = {"FUND": "#2E7D32", "SKIP": "#C62828"}
bar_colors = [color_map.get(r, "#90A4AE") for r in top10["Recommend"]]

fig_corp = go.Figure()
fig_corp.add_trace(go.Bar(
    x=top10["Label"],
    y=top10["EV_M"],
    marker_color=bar_colors,
    text=[f"₹{v:.1f}M<br>{r}" for v, r in zip(top10["EV_M"], top10["Recommend"])],
    textposition="outside",
    textfont=dict(size=10),
    hovertemplate="<b>%{x}</b><br>Risk-Adj EV: ₹%{y:.2f}M<br>Dept: %{customdata[0]}<br>Risk: %{customdata[1]}<extra></extra>",
    customdata=top10[["Department","Project_Risk"]].values,
    name="Risk-Adj EV"
))
fig_corp.add_trace(go.Scatter(
    x=top10["Label"],
    y=top10["NPV_M"],
    mode="markers",
    marker=dict(symbol="diamond", size=10, color="#1565C0", line=dict(color="white", width=1)),
    name="NPV",
    hovertemplate="NPV: ₹%{y:.2f}M<extra></extra>"
))
fig_corp.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
fig_corp.update_layout(
    title=dict(text="Top 10 Projects — Risk-Adjusted Expected Value<br><sup>Green=FUND | Red=SKIP | Diamonds=NPV</sup>",
               font=dict(size=15)),
    xaxis_title="Project ID",
    yaxis_title="Risk-Adjusted EV (₹ Millions)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    template="plotly_white",
    height=450
)

# Fund vs Skip summary
fund_count = (m1_ranked["Recommend"] == "FUND").sum()
skip_count = (m1_ranked["Recommend"] == "SKIP").sum()
total_inv  = m1_ranked[m1_ranked["Recommend"]=="FUND"]["Investment_Cost"].sum()
print(f"Funding recommendations: FUND={fund_count} | SKIP={skip_count}")
print(f"Total capital to deploy: ₹{total_inv/1e6:.1f}M")
fig_corp.show()

Funding recommendations: FUND=16 | SKIP=34
Total capital to deploy: ₹31.6M


---
## 3. Panel 2 — Banking: Credit Risk & Fraud Detection

In [5]:
# Credit risk tier donut
tier_counts = m2_loans["PD_Tier"].value_counts().reset_index()
tier_counts.columns = ["Tier", "Count"]
tier_order = {"Low":0,"Medium":1,"High":2}
tier_counts["order"] = tier_counts["Tier"].map(tier_order)
tier_counts = tier_counts.sort_values("order")

tier_colors = {"Low":"#2E7D32","Medium":"#F57F17","High":"#C62828"}
avg_pd_by_tier = m2_loans.groupby("PD_Tier")["PD_Predicted"].mean()

fig_bank = make_subplots(
    rows=1, cols=2,
    specs=[[{"type":"pie"},{"type":"table"}]],
    subplot_titles=["Credit Risk Tier Distribution","Fraud Alert Summary"]
)

fig_bank.add_trace(go.Pie(
    labels=tier_counts["Tier"],
    values=tier_counts["Count"],
    hole=0.55,
    marker_colors=[tier_colors[t] for t in tier_counts["Tier"]],
    textinfo="label+percent",
    textfont=dict(size=13),
    hovertemplate="<b>%{label}</b><br>Count: %{value}<br>Share: %{percent}<extra></extra>"
), row=1, col=1)

# Fraud table — show confirmed frauds
confirmed_fraud = m2_fraud[m2_fraud["Fraud_Predicted"]==1][
    ["Transaction_ID","Customer_ID","Amount","Transaction_Type","Fraud_Score"]
].copy()
confirmed_fraud = confirmed_fraud.sort_values("Fraud_Score", ascending=False).head(10)
confirmed_fraud["Fraud_Score"] = confirmed_fraud["Fraud_Score"].round(3)
confirmed_fraud["Amount"] = confirmed_fraud["Amount"].apply(lambda x: f"₹{x:,.0f}")

fig_bank.add_trace(go.Table(
    header=dict(
        values=["<b>Txn ID</b>","<b>Cust ID</b>","<b>Amount</b>","<b>Type</b>","<b>Fraud Score</b>"],
        fill_color="#B71C1C",
        font=dict(color="white", size=11),
        align="center",
        height=28
    ),
    cells=dict(
        values=[confirmed_fraud["Transaction_ID"], confirmed_fraud["Customer_ID"],
                confirmed_fraud["Amount"], confirmed_fraud["Transaction_Type"],
                confirmed_fraud["Fraud_Score"]],
        fill_color=[["#FFEBEE","white"]*10],
        align=["center","center","right","center","center"],
        font=dict(size=10),
        height=24
    )
), row=1, col=2)

fig_bank.update_layout(
    title=dict(text="Banking Risk Dashboard — Credit Tiers & Fraud Alerts", font=dict(size=15)),
    template="plotly_white",
    height=420,
    annotations=[
        dict(text=f"<b>{len(m2_loans)}</b><br>Total Loans", x=0.18, y=0.5,
             font=dict(size=13), showarrow=False),
    ]
)
total_fraud = (m2_fraud["Fraud_Predicted"]==1).sum()
print(f"Fraud alerts raised: {total_fraud} out of {len(m2_fraud)} transactions")
print(f"Credit tiers — Low: {tier_counts[tier_counts.Tier=='Low']['Count'].values[0]} | "
      f"Medium: {tier_counts[tier_counts.Tier=='Medium']['Count'].values[0]} | "
      f"High: {tier_counts[tier_counts.Tier=='High']['Count'].values[0]}")
fig_bank.show()

Fraud alerts raised: 12 out of 200 transactions
Credit tiers — Low: 18 | Medium: 63 | High: 19


---
## 4. Panel 3 — Financial Markets: Portfolio & Backtest

In [6]:
# Max-Sharpe allocation pie
w_ms = m3_weights[["ticker","max_sharpe_weight"]].copy()
w_ms["max_sharpe_weight"] = w_ms["max_sharpe_weight"].clip(lower=1e-6)
w_ms = w_ms[w_ms["max_sharpe_weight"] > 0.001]

stock_colors = {"ASIANPAINT":"#1565C0","CIPLA":"#2E7D32","HDFCBANK":"#6A1B9A",
                "ITC":"#E65100","TCS":"#00695C","ULTRACEMCO":"#AD1457"}

fig_mkt = make_subplots(
    rows=1, cols=2,
    specs=[[{"type":"pie"},{"type":"scatter"}]],
    subplot_titles=["Max-Sharpe Portfolio Allocation","Backtest: ML vs Equal-Weight (2024–2026)"]
)

# Add predicted return to label
m3_stocks_idx = m3_stocks.set_index("ticker")
labels_pie = [
    f"{t}<br>Wt:{w:.1%}<br>PredRet:{m3_stocks_idx.loc[t,'Ann_Pred_Return']:.1%}"
    if t in m3_stocks_idx.index else f"{t}<br>Wt:{w:.1%}"
    for t, w in zip(w_ms["ticker"], w_ms["max_sharpe_weight"])
]
fig_mkt.add_trace(go.Pie(
    labels=w_ms["ticker"],
    values=w_ms["max_sharpe_weight"],
    hole=0.4,
    marker_colors=[stock_colors.get(t,"#90A4AE") for t in w_ms["ticker"]],
    text=labels_pie,
    textinfo="label+percent",
    textfont=dict(size=11),
    hovertemplate="<b>%{label}</b><br>Weight: %{percent}<extra></extra>"
), row=1, col=1)

# Backtest equity curve
fig_mkt.add_trace(go.Scatter(
    x=m3_backtest["date"], y=m3_backtest["ml_cum"],
    mode="lines", name="ML Max-Sharpe",
    line=dict(color="#2E7D32", width=2.5),
    hovertemplate="%{x|%b %Y}<br>Value: %{y:.3f}<extra>ML</extra>"
), row=1, col=2)
fig_mkt.add_trace(go.Scatter(
    x=m3_backtest["date"], y=m3_backtest["eq_cum"],
    mode="lines", name="Equal-Weight",
    line=dict(color="#90A4AE", width=2, dash="dash"),
    hovertemplate="%{x|%b %Y}<br>Value: %{y:.3f}<extra>EW</extra>"
), row=1, col=2)
fig_mkt.add_trace(go.Scatter(
    x=[m3_backtest["date"].min(), m3_backtest["date"].max()],
    y=[1.0, 1.0], mode="lines",
    line=dict(color="black", width=1, dash="dot"),
    showlegend=False, hoverinfo="skip"
), row=1, col=2)

fig_mkt.update_layout(
    title=dict(text="Financial Markets — Portfolio Allocation & Backtest Performance", font=dict(size=15)),
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.75)
)
fig_mkt.update_yaxes(title_text="Portfolio Value (base=1.0)", row=1, col=2)
fig_mkt.update_xaxes(title_text="Date", row=1, col=2)

ml_final  = m3_backtest["ml_cum"].iloc[-1]
eq_final  = m3_backtest["eq_cum"].iloc[-1]
print(f"Backtest final values — ML: {ml_final:.3f} | EW: {eq_final:.3f}")
fig_mkt.show()

Backtest final values — ML: 0.601 | EW: 0.861


---
## 5. Panel 4 — Derivatives: Pricing, VaR & Hedge

In [7]:
fig_deriv = make_subplots(
    rows=1, cols=3,
    specs=[[{"type":"bar"},{"type":"indicator"},{"type":"table"}]],
    subplot_titles=["Option Pricing: BS vs ML RMSE","VaR 99% (₹1 Cr NIFTY)","Delta Hedge Summary"],
    column_widths=[0.35, 0.30, 0.35]
)

# Pricing comparison bar
model_rmse = m4_pricing["RMSE_INR"].tolist()
model_names = m4_pricing["Model"].tolist()
bar_cols = ["#FF7043","#5C6BC0","#1565C0"]
fig_deriv.add_trace(go.Bar(
    x=model_names,
    y=model_rmse,
    marker_color=bar_cols,
    text=[f"{v:.2f}" for v in model_rmse],
    textposition="outside",
    textfont=dict(size=12),
    hovertemplate="<b>%{x}</b><br>RMSE: ₹%{y:.2f}<extra></extra>",
    showlegend=False
), row=1, col=1)
fig_deriv.update_yaxes(title_text="RMSE (₹ INR)", row=1, col=1)

# VaR gauge
var_99_val = m4_var[m4_var["Metric"]=="VaR_99"]["INR_1cr"].values[0]
cvar_99_val = m4_var[m4_var["Metric"]=="CVaR_99"]["INR_1cr"].values[0]
fig_deriv.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=var_99_val,
    number=dict(prefix="₹", suffix="", valueformat=",.0f"),
    delta=dict(reference=cvar_99_val, increasing=dict(color="#C62828"),
               decreasing=dict(color="#2E7D32"), valueformat=",.0f",
               prefix="CVaR: ₹"),
    title=dict(text="<b>VaR 99%</b><br><sup>5-min intraday loss</sup>", font=dict(size=12)),
    gauge=dict(
        axis=dict(range=[0, cvar_99_val * 1.5], tickformat=",.0f"),
        bar=dict(color="#E53935"),
        steps=[
            dict(range=[0, var_99_val * 0.5],  color="#C8E6C9"),
            dict(range=[var_99_val * 0.5, var_99_val], color="#FFF9C4"),
            dict(range=[var_99_val, cvar_99_val * 1.5], color="#FFCDD2"),
        ],
        threshold=dict(line=dict(color="#B71C1C", width=3), thickness=0.75, value=var_99_val)
    )
), row=1, col=2)

# Hedge summary table
fig_deriv.add_trace(go.Table(
    header=dict(
        values=["<b>Parameter</b>","<b>Value</b>"],
        fill_color="#1A237E",
        font=dict(color="white", size=11),
        align=["left","center"],
        height=28
    ),
    cells=dict(
        values=[m4_hedge["Parameter"], m4_hedge["Value"]],
        fill_color=[["#E8EAF6","white"]*len(m4_hedge)],
        align=["left","center"],
        font=dict(size=10),
        height=23
    )
), row=1, col=3)

fig_deriv.update_layout(
    title=dict(text="Derivatives — Option Pricing Accuracy | VaR | Delta Hedge", font=dict(size=15)),
    template="plotly_white",
    height=440
)
fig_deriv.show()

---
## 6. Full Integrated Dashboard

All four panels combined into a single scrollable executive report.

In [8]:
from plotly.subplots import make_subplots

fig_full = make_subplots(
    rows=4, cols=2,
    specs=[
        [{"type":"xy",    "colspan":2}, None],
        [{"type":"pie"},  {"type":"table"}],
        [{"type":"pie"},  {"type":"scatter"}],
        [{"type":"bar"},  {"type":"table"}],
    ],
    subplot_titles=[
        "M1: Top 10 Projects — Risk-Adjusted Expected Value (₹M)",
        "M2: Credit Risk Tiers", "M2: Fraud Alerts (Top 10 by Score)",
        "M3: Max-Sharpe Portfolio", "M3: Backtest Equity Curve (2024–2026)",
        "M4: Option Pricing RMSE", "M4: Delta Hedge Summary"
    ],
    vertical_spacing=0.10,
    horizontal_spacing=0.08,
    row_heights=[0.22, 0.24, 0.27, 0.27]
)

# ─── Row 1: Corporate Finance bar ───────────────────────────────────────────
fig_full.add_trace(go.Bar(
    x=top10["Label"], y=top10["EV_M"],
    marker_color=bar_colors, showlegend=False,
    name="Risk-Adj EV",
    text=[f"₹{v:.1f}M" for v in top10["EV_M"]],
    textposition="outside", textfont=dict(size=9),
    hovertemplate="<b>%{x}</b><br>EV: ₹%{y:.2f}M<br>Dept: %{customdata[0]}<extra></extra>",
    customdata=top10[["Department"]].values
), row=1, col=1)
fig_full.update_yaxes(title_text="Risk-Adj EV (₹M)", row=1, col=1)

# ─── Row 2: Credit risk pie ──────────────────────────────────────────────────
fig_full.add_trace(go.Pie(
    labels=tier_counts["Tier"],
    values=tier_counts["Count"],
    hole=0.5,
    marker_colors=[tier_colors[t] for t in tier_counts["Tier"]],
    textinfo="label+percent",
    textfont=dict(size=11),
    showlegend=False
), row=2, col=1)

# Fraud table (top 8)
fig_full.add_trace(go.Table(
    header=dict(values=["<b>Txn</b>","<b>Cust</b>","<b>Amount</b>","<b>Type</b>","<b>Score</b>"],
                fill_color="#B71C1C", font=dict(color="white",size=10),
                align="center", height=24),
    cells=dict(
        values=[confirmed_fraud["Transaction_ID"].head(8),
                confirmed_fraud["Customer_ID"].head(8),
                confirmed_fraud["Amount"].head(8),
                confirmed_fraud["Transaction_Type"].head(8),
                confirmed_fraud["Fraud_Score"].head(8)],
        fill_color=[["#FFEBEE","white"]*8],
        align=["center","center","right","center","center"],
        font=dict(size=9), height=20)
), row=2, col=2)

# ─── Row 3: Markets pie + equity curve ───────────────────────────────────────
fig_full.add_trace(go.Pie(
    labels=w_ms["ticker"], values=w_ms["max_sharpe_weight"],
    hole=0.4,
    marker_colors=[stock_colors.get(t,"#90A4AE") for t in w_ms["ticker"]],
    textinfo="label+percent", textfont=dict(size=10),
    showlegend=False
), row=3, col=1)

fig_full.add_trace(go.Scatter(
    x=m3_backtest["date"], y=m3_backtest["ml_cum"],
    mode="lines", line=dict(color="#2E7D32", width=2), name="ML",
    showlegend=True
), row=3, col=2)
fig_full.add_trace(go.Scatter(
    x=m3_backtest["date"], y=m3_backtest["eq_cum"],
    mode="lines", line=dict(color="#90A4AE", width=2, dash="dash"), name="EW",
    showlegend=True
), row=3, col=2)
fig_full.update_yaxes(title_text="Portfolio Value", row=3, col=2)

# ─── Row 4: Derivatives bar + hedge table ────────────────────────────────────
fig_full.add_trace(go.Bar(
    x=model_names, y=model_rmse,
    marker_color=bar_cols, showlegend=False,
    text=[f"{v:.1f}" for v in model_rmse],
    textposition="outside", textfont=dict(size=9)
), row=4, col=1)
fig_full.update_yaxes(title_text="RMSE (₹)", row=4, col=1)

fig_full.add_trace(go.Table(
    header=dict(values=["<b>Parameter</b>","<b>Value</b>"],
                fill_color="#1A237E", font=dict(color="white",size=10),
                align=["left","center"], height=24),
    cells=dict(
        values=[m4_hedge["Parameter"], m4_hedge["Value"]],
        fill_color=[["#E8EAF6","white"]*len(m4_hedge)],
        align=["left","center"], font=dict(size=9), height=20)
), row=4, col=2)

fig_full.update_layout(
    title=dict(
        text="<b>Aurora Finance — ML-Driven Decision Support Dashboard</b><br>"
             "<sup>Modules: Corporate Finance | Banking | Financial Markets | Derivatives</sup>",
        font=dict(size=18), x=0.5
    ),
    template="plotly_white",
    height=1400,
    showlegend=True,
    legend=dict(orientation="h", yanchor="top", y=-0.02, xanchor="center", x=0.5)
)

fig_full.show()
print("Full integrated dashboard rendered.")

Full integrated dashboard rendered.


---
## 7. Export — Self-Contained HTML Dashboard

In [9]:
html_path = os.path.join(_nb_dir, "aurora_dashboard.html")
fig_full.write_html(html_path, include_plotlyjs="cdn", full_html=True)
print(f"Saved interactive HTML: {html_path}")
print(f"File size: {os.path.getsize(html_path)/1024:.1f} KB")

Saved interactive HTML: D:\IIT Course\Sem3\iitb-capstone-project\Dashboard\aurora_dashboard.html
File size: 56.8 KB


---
## 8. Export — Executive Excel Summary Report

In [10]:
excel_path = os.path.join(_nb_dir, "aurora_executive_summary.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    # Sheet 1: Corporate Finance — Top 10 Projects
    corp_export = top10[["Label","Department","Project_Risk","NPV_M","EV_M","Recommend"]].copy()
    corp_export.columns = ["Project","Department","Risk","NPV (₹M)","Risk-Adj EV (₹M)","Decision"]
    corp_export.to_excel(writer, sheet_name="M1 Corp Finance", index=False)

    # Sheet 2: Banking — Credit Risk Summary
    tier_summary = m2_loans.groupby("PD_Tier").agg(
        Count=("Loan_ID","count"),
        Avg_PD=("PD_Predicted","mean"),
        Total_Loan_Amt=("Loan_Amount","sum")
    ).reset_index()
    tier_summary["Avg_PD"] = tier_summary["Avg_PD"].round(4)
    tier_summary["Total_Loan_Amt"] = tier_summary["Total_Loan_Amt"].round(0)
    tier_summary.to_excel(writer, sheet_name="M2 Credit Risk", index=False)

    # Sheet 3: Banking — Fraud alerts
    fraud_export = m2_fraud[m2_fraud["Fraud_Predicted"]==1][
        ["Transaction_ID","Customer_ID","Amount","Transaction_Type","Fraud_Score"]
    ].sort_values("Fraud_Score", ascending=False).copy()
    fraud_export.to_excel(writer, sheet_name="M2 Fraud Alerts", index=False)

    # Sheet 4: Financial Markets — Stock ranking + weights
    mkt_export = m3_stocks.copy()
    mkt_export = mkt_export.merge(
        m3_weights[["ticker","max_sharpe_weight","min_var_weight","equal_weight"]],
        on="ticker", how="left"
    )
    mkt_export["Ann_Pred_Return"] = mkt_export["Ann_Pred_Return"].round(4)
    mkt_export["Ann_Volatility"]  = mkt_export["Ann_Volatility"].round(4)
    mkt_export["Pred_Sharpe"]     = mkt_export["Pred_Sharpe"].round(3)
    mkt_export.to_excel(writer, sheet_name="M3 Markets", index=False)

    # Sheet 5: Backtest summary
    bt_summary = pd.DataFrame({
        "Strategy": ["ML Max-Sharpe", "Equal-Weight"],
        "Start Value": [1.0, 1.0],
        "End Value": [round(m3_backtest["ml_cum"].iloc[-1],4),
                      round(m3_backtest["eq_cum"].iloc[-1],4)],
        "Ann Return": [
            round((m3_backtest["ml_ret"].mean()*252),4),
            round((m3_backtest["eq_ret"].mean()*252),4)
        ],
        "Sharpe (approx)": [
            round((m3_backtest["ml_ret"].mean()-0.065/252)/(m3_backtest["ml_ret"].std()+1e-10)*np.sqrt(252),3),
            round((m3_backtest["eq_ret"].mean()-0.065/252)/(m3_backtest["eq_ret"].std()+1e-10)*np.sqrt(252),3)
        ]
    })
    bt_summary.to_excel(writer, sheet_name="M3 Backtest", index=False)

    # Sheet 6: Derivatives
    deriv_export = m4_pricing.copy()
    deriv_export.to_excel(writer, sheet_name="M4 Option Pricing", index=False)
    m4_var.to_excel(writer, sheet_name="M4 VaR", index=False, startrow=0)
    m4_hedge.to_excel(writer, sheet_name="M4 Delta Hedge", index=False)

print(f"Saved Excel summary: {excel_path}")
print(f"File size: {os.path.getsize(excel_path)/1024:.1f} KB")

Saved Excel summary: D:\IIT Course\Sem3\iitb-capstone-project\Dashboard\aurora_executive_summary.xlsx
File size: 10.5 KB


---
## 9. Executive Narrative — Key Findings Across Modules

In [11]:
fund_count = (m1_ranked["Recommend"] == "FUND").sum()
total_ev   = m1_ranked[m1_ranked["Recommend"]=="FUND"]["Risk_Adj_EV"].sum() / 1e6
low_risk   = (m2_loans["PD_Tier"]=="Low").sum()
high_risk  = (m2_loans["PD_Tier"]=="High").sum()
fraud_count= (m2_fraud["Fraud_Predicted"]==1).sum()
top_stock  = m3_stocks.iloc[0]["ticker"]
top_return = m3_stocks.iloc[0]["Ann_Pred_Return"]
ms_wt_top  = m3_weights.loc[m3_weights["max_sharpe_weight"].idxmax()]
bs_rmse    = m4_pricing[m4_pricing["Model"]=="Black-Scholes"]["RMSE_INR"].values[0]
ml_rmse    = m4_pricing[m4_pricing["Model"]=="XGBoost Set B"]["RMSE_INR"].values[0]
var_99     = m4_var[m4_var["Metric"]=="VaR_99"]["INR_1cr"].values[0]

print("=" * 72)
print("   AURORA FINANCE — EXECUTIVE SUMMARY (All Modules)")
print("=" * 72)
print()
print("MODULE 1 — CORPORATE FINANCE (Project Funding)")
print(f"  Recommendation  : FUND {fund_count}/50 projects")
print(f"  Total ML-EV     : ₹{total_ev:.1f}M across funded projects")
print(f"  Top project     : {top10.iloc[0]['Label']} ({top10.iloc[0]['Department']}) "
      f"EV=₹{top10.iloc[0]['EV_M']:.1f}M")
print(f"  Key driver (SHAP): NPV and Payback_Ratio dominate success probability")
print()
print("MODULE 2 — BANKING (Credit Risk & Fraud)")
print(f"  Credit risk     : Low={low_risk} | Med={(len(m2_loans)-low_risk-high_risk)} | High={high_risk} loans")
print(f"  Fraud alerts    : {fraud_count} flagged transactions out of {len(m2_fraud)}")
print(f"  Best model      : XGBoost with SMOTE (Precision-Recall AUC > baseline)")
print()
print("MODULE 3 — FINANCIAL MARKETS (Portfolio)")
print(f"  Top ranked stock: {top_stock} (Pred. Ann. Return = {top_return:.1%})")
print(f"  Max-Sharpe top wt: {ms_wt_top['ticker']} ({ms_wt_top['max_sharpe_weight']:.1%})")
print(f"  Backtest (2024–26): ML={m3_backtest['ml_cum'].iloc[-1]:.3f} vs EW={m3_backtest['eq_cum'].iloc[-1]:.3f}")
print()
print("MODULE 4 — DERIVATIVES (Options & Risk)")
print(f"  BS RMSE         : ₹{bs_rmse:.2f} → ML RMSE: ₹{ml_rmse:.2f} "
      f"({'ML better' if ml_rmse < bs_rmse else 'BS better'})")
print(f"  VaR 99% (₹1cr) : ₹{var_99:,.0f} per 5-min interval")
print(f"  Delta hedge     : {m4_hedge[m4_hedge.Parameter=='Contracts (lots)']['Value'].values[0]} NIFTY puts @ "
      f"{m4_hedge[m4_hedge.Parameter=='Hedge Instrument']['Value'].values[0].split('@')[1].strip()}")
print()
print("Dashboard files:")
print(f"  aurora_dashboard.html")
print(f"  aurora_executive_summary.xlsx")
print("=" * 72)

   AURORA FINANCE — EXECUTIVE SUMMARY (All Modules)

MODULE 1 — CORPORATE FINANCE (Project Funding)
  Recommendation  : FUND 16/50 projects
  Total ML-EV     : ₹24.5M across funded projects
  Top project     : P49 (IT) EV=₹3.3M
  Key driver (SHAP): NPV and Payback_Ratio dominate success probability

MODULE 2 — BANKING (Credit Risk & Fraud)
  Credit risk     : Low=18 | Med=63 | High=19 loans
  Fraud alerts    : 12 flagged transactions out of 200
  Best model      : XGBoost with SMOTE (Precision-Recall AUC > baseline)

MODULE 3 — FINANCIAL MARKETS (Portfolio)
  Top ranked stock: ITC (Pred. Ann. Return = 32.4%)
  Max-Sharpe top wt: ITC (68.9%)
  Backtest (2024–26): ML=0.601 vs EW=0.861

MODULE 4 — DERIVATIVES (Options & Risk)
  BS RMSE         : ₹29.57 → ML RMSE: ₹15.15 (ML better)
  VaR 99% (₹1cr) : ₹34,156 per 5-min interval
  Delta hedge     : 15 NIFTY puts @ 21850

Dashboard files:
  aurora_dashboard.html
  aurora_executive_summary.xlsx
